In [1]:
# Pass marks, set before the run.
MAX_LLM_COST_USD = 12.0  # stop before going past this on LLM calls
MAX_MALFORMED_SHARE = 0.05  # replies that are not the JSON asked for, per method
# Which version 2.0.0 run of the authors' HumBERT recipe saved its rankings (notebook 13's ran without them).
RECIPE_RUN = "full_v2_rankings"

# 15 · Every HumSet method on HumSet 2.0.0

HumBERT's published results were scored on HumSet 2.0.0, the version its authors' code loads; every other HumSet
result here uses version 1.0.0, the dataset the HumSet paper describes (notebook 13). The two versions split the
data differently: only 1,252 test excerpts are in both, and 78% of version 2.0.0's test excerpts are in version
1.0.0's training split. So no model trained on 1.0.0 can be scored on 2.0.0, and this notebook runs every method
again, on 2.0.0's own splits, stored under the test set name `humset_v2` so that nothing mixes with 1.0.0.

Every method is run as its own notebook ran it on 1.0.0: the same settings, the same sample size (500 test
excerpts in each of English, French and Spanish), the same tuning items, and the same scoring, exact and tier by
tier, as notebook 14, and on the full test set by macro F1 per published category, as notebook 10. The fine-tuned
models come from the Hugging Face jobs in `scripts/hf_jobs/launch_humset_v2.sh`, with their version 1.0.0
arguments. The combined method chooses its sources on development excerpts from the same nine candidates as on
1.0.0 (notebooks 10 and 11).

Differences from 1.0.0: version 2.0.0 has 66 labels in use against 62 (two renamed, four new), no project id
(its project title stands in), and one Portuguese test excerpt, which the sample leaves out and the full test
set keeps. Version 1.0.0 keeps each source document's excerpts in one split, and version 2.0.0 does not; the
last cell measures how many test excerpts share a source document with training in each version.

In [2]:
import json

import numpy as np
import pandas as pd

from tiered_taxonomy import benchmarks as bm
from tiered_taxonomy import experiment as ex
from tiered_taxonomy import gpu_jobs as gj
from tiered_taxonomy import humbert_reproduction as hr
from tiered_taxonomy import like_for_like as lf
from tiered_taxonomy import new_technique as nt
from tiered_taxonomy import runs
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.methods import combined as cb
from tiered_taxonomy.methods.trained import choose_cut

root = data_root()
config = load_config()
seed = config["seed"]
words = config["text"]["input_words"]
trained = config["trained"]
limits, thresholds = trained["cut_limits"], trained["cut_thresholds"]
plan = config["new_technique"]
data = ex.load_humset(root, ex.HUMSET_V2)
sample = ex.sample_humset(data, config["samples"]["humset"]["per_language"], seed)
tuning = ex.pilot_subset(data.dev, config["pipeline"]["tuning_items"], seed)
bench = runs.make_bench(data, root, config)
runs.load_cards(bench)
runs.embed_inputs(bench, sample)
results = {
    "sizes": {"train": len(data.train), "dev": len(data.dev), "test": len(data.test), "sample": len(sample)}
}
print(results["sizes"], "labels:", len(data.taxonomy.nodes))

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'train': 129268, 'dev': 11940, 'test': 12896, 'sample': 1500} labels: 79


## The LLM methods, on the sample

As in notebooks 03 to 06 and 09. Each method's cost is checked against the limit as it goes.

In [3]:
spent = 0.0
llm = {}


def charge(name, run):
    global spent
    spent += run["cost_usd"]
    share = run["malformed"] / run["calls"] if run["calls"] else 0.0
    print(f"{name}: ${run['cost_usd']:.2f}, {run['calls']} calls, malformed {share:.1%}; spent ${spent:.2f}")
    if share > MAX_MALFORMED_SHARE:
        raise RuntimeError(f"{name}: {share:.1%} of replies were malformed")
    if spent > MAX_LLM_COST_USD:
        raise RuntimeError(f"LLM calls have cost ${spent:.2f}, over the ${MAX_LLM_COST_USD:.2f} limit")


def keep(name, outcome):
    run = outcome["run"]
    charge(name, run)
    llm[name] = {
        "scores": outcome["scores"],
        "cost_per_1000": round(run["cost_usd"] / run["items"] * 1000, 2),
        "calls_per_item": round(run["calls"] / run["items"], 2),
    }


pipeline_tuning = runs.tune_pipeline(bench, tuning)
for method in (
    "pipeline",
    "select_all",
    "funnel_names",
    "funnel_cards",
    "walk_strict",
    "walk_branching",
    "walk_anchored",
):
    keep(method, runs.run_and_score(bench, method, sample))
for method, outcome in runs.evidence_runs(bench, "funnel_cards", sample).items():
    if method != "funnel_cards":
        keep(method, outcome)

pipeline: $0.50, 4500 calls, malformed 0.0%; spent $0.50


select_all: $0.22, 1500 calls, malformed 0.0%; spent $0.72


funnel_names: $0.28, 1500 calls, malformed 0.0%; spent $1.00


funnel_cards: $0.44, 1500 calls, malformed 0.0%; spent $1.44
walk_strict: $0.20, 2195 calls, malformed 0.0%; spent $1.64


walk_branching: $0.24, 2603 calls, malformed 0.0%; spent $1.88


walk_anchored: $0.35, 3277 calls, malformed 0.0%; spent $2.22


funnel_cards_quote_filter: $0.44, 1500 calls, malformed 0.0%; spent $2.66
funnel_cards_checked: $0.59, 2993 calls, malformed 0.0%; spent $3.26


In [4]:
recipe_bench = runs.make_bench(data, root, config)
recipe_bench.workers = config["paper_models"]["workers"]
tools = runs.make_recipe_tools(recipe_bench)
ids = [item.id for item in sample]
for recipe in ("llm_ensemble", "ki_fsprompt"):
    dev_ranked, dev_record = runs.run_recipe(recipe_bench, tools, recipe, "dev", tuning)
    charge(f"{recipe} tuning", dev_record.__dict__)
    cut, _ = choose_cut(ex.gold_exact(tuning), dev_ranked, limits, thresholds)
    ranked, record = runs.run_recipe(recipe_bench, tools, recipe, "test", sample)
    labels = [cut.apply(ranking) for ranking in ranked]
    ex.save_predictions(root, data.name, recipe, ids, labels)
    keep(recipe, {"scores": ex.evaluate(data, sample, labels), "run": record.__dict__})
results["llm"] = llm
results["llm_cost_usd"] = round(spent, 2)
pd.DataFrame(
    {
        name: {"exact F1": row["scores"]["exact"]["f1"], "$ per 1,000": row["cost_per_1000"]}
        for name, row in llm.items()
    }
).T.sort_values("exact F1", ascending=False).round(3)

llm_ensemble tuning: $0.57, 2200 calls, malformed 0.5%; spent $3.83


llm_ensemble: $4.26, 16500 calls, malformed 0.3%; spent $8.09


ki_fsprompt tuning: $0.03, 600 calls, malformed 0.3%; spent $8.11


ki_fsprompt: $0.23, 4500 calls, malformed 0.3%; spent $8.34


,exact F1,"$ per 1,000"
ki_fsprompt,0.563,0.15
walk_branching,0.396,0.16
select_all,0.375,0.15
funnel_cards,0.352,0.29
walk_strict,0.351,0.13
funnel_names,0.348,0.18
pipeline,0.330,0.33
funnel_cards_checked,0.327,0.40
llm_ensemble,0.310,2.84
funnel_cards_quote_filter,0.302,0.29


## The models trained on this machine

Omikuji, the embedding classifier, similar documents and label search, as in notebook 08, from BGE-M3 embeddings
of every excerpt (`scripts/embed_corpus.py humset_v2`).

In [5]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
details = sg.collect_signals(data, root, trained, seed, words, device, bench.index)
print(sorted(details))

['embedding_classifier', 'knn', 'label_search', 'omikuji']


## The fine-tuned models, from their Hugging Face jobs

In [6]:
FETCHED = {
    "classifier_humset_v2": "classifier",
    "reranker_humset_v2": "reranker",
    "llm_humset_v2": "llm",
    "gliner_humset_v2": "gliner",
}
dropped = {
    run: len(nt.fetch_fine_tuned(root, data.name, run, source, data.taxonomy))
    for run, source in FETCHED.items()
}
dropped["xr_transformer"] = len(sg.xr_signals(data, root))
for split, items, name in (
    ("dev", data.dev, "rankings_validation.jsonl"),
    ("test", data.test, "rankings_test.jsonl"),
):
    path = gj.fetch(root, f"humbert_reproduction/{RECIPE_RUN}/{name}")
    rankings = hr.to_rankings([json.loads(line) for line in open(path, encoding="utf-8")])
    missing = {item.id for item in items} - set(rankings)
    if missing:
        raise ValueError(f"humbert_recipe {split}: {len(missing)} excerpts have no ranking")
    sg.save_signal(root, data.name, "humbert_recipe", split, rankings)
print("labels dropped for having no place in the tree:", dropped)

labels dropped for having no place in the tree: {'classifier_humset_v2': 0, 'reranker_humset_v2': 0, 'llm_humset_v2': 0, 'gliner_humset_v2': 0, 'xr_transformer': 0}


## Every ranked model on the sample, and the combined method

The trained and fine-tuned models are cut and scored as in notebook 14, with the cut chosen on the same 4,000
development excerpts the combined method cuts on.

In [7]:
order = np.random.default_rng(seed).permutation(len(data.dev))
raw = [data.dev[int(i)] for i in order[: plan["humset"]["fit_items"] + plan["humset"]["cut_items"]]]
fit, cut_items = (
    nt.with_all_tiers(raw[: plan["humset"]["fit_items"]]),
    nt.with_all_tiers(raw[plan["humset"]["fit_items"] :]),
)
tune = raw[plan["humset"]["fit_items"] :]
SOURCES = [
    "classifier",
    "llm",
    "reranker",
    "gliner",
    "xr_transformer",
    "omikuji",
    "embedding_classifier",
    "knn",
    "label_search",
    "humbert_recipe",
]
dev_sources = sg.load_sources(root, data.name, SOURCES, "dev")
test_sources = sg.load_sources(root, data.name, SOURCES, "test")
ranked = {}
for name in SOURCES:
    merged = {**dev_sources[name], **test_sources[name]}
    made = lf.score_ranked(
        data,
        tune,
        lf.source_rankings(merged, tune, name),
        sample,
        lf.source_rankings(merged, sample, name),
        limits,
        thresholds,
    )
    made.pop("labels")
    ranked[name] = {"family": "ranked", **made}
results["ranked"] = ranked
pd.DataFrame(
    {
        name: {"exact F1": row["scores"]["exact"]["f1"], "limit": row["limit"], "cut-off": row["threshold"]}
        for name, row in ranked.items()
    }
).T.sort_values("exact F1", ascending=False).round(3)

,exact F1,limit,cut-off
xr_transformer,0.686,4.0,0.35
knn,0.661,5.0,0.40
humbert_recipe,0.659,15.0,0.50
classifier,0.648,10.0,0.50
omikuji,0.641,4.0,0.20
embedding_classifier,0.596,3.0,0.30
reranker,0.513,5.0,0.60
llm,0.482,10.0,0.40
gliner,0.482,6.0,0.50
label_search,0.101,8.0,0.00


In [8]:
CANDIDATES = [
    "classifier",
    "xr_transformer",
    "reranker",
    "llm",
    "omikuji",
    "embedding_classifier",
    "knn",
    "label_search",
    "gliner",
]
settings = cb.CombinedSettings(per_source=config["combined"]["per_source"], seed=seed)
half = len(cut_items) // 2
h_dev = nt.expand_sources(sg.load_sources(root, data.name, CANDIDATES, "dev"))
graph = cb.LabelGraph(data.taxonomy, [sorted(bm.gold_with_parents(item)) for item in data.train])
choice = nt.development_choice(
    "humset", fit, cut_items, cut_items[:half], cut_items[half:], h_dev, graph, words, settings, data.taxonomy
)
names = choice["chosen"]
best = nt.best_single("humset", choice)
print("chosen on development excerpts:", names, "| best single source:", best)
results["choice"] = {key: choice[key] for key in ("single", "combined", "chosen", "chosen_scores")}

chosen on development excerpts: ['classifier', 'knn'] | best single source: classifier


In [9]:
test = nt.with_all_tiers(data.test)
h_test = nt.expand_sources(sg.load_sources(root, data.name, sorted({*names, best}), "test"))
stack = nt.Stack(names, fit, cut_items, h_dev, h_test, graph, words, settings)
final = nt.humset_final(stack, test, best, data.taxonomy, seed)
final.pop("rankings")
results["full_test"] = {**final, "sources": names, "best_single_source": best}
exact_labels = {label for item in data.train for label in item.labels}
tune_rankings, sample_rankings = lf.combined_rankings(stack, sample)
made = lf.score_ranked(
    data,
    tune,
    lf.restrict(tune_rankings, exact_labels),
    sample,
    lf.restrict(sample_rankings, exact_labels),
    limits,
    thresholds,
)
made.pop("labels")
results["combined"] = {"combined": {"family": "combined", "sources": list(names), "priors": False, **made}}
printed = {**bm.HUMBERT_BEST, "Mean": float(np.mean(list(bm.HUMBERT_BEST.values())))}
pd.DataFrame(
    {
        "combined method": final["scores"],
        "best single source": final["best_single"],
        "HumBERT, printed": printed,
    }
).round(3)

,combined method,best single source,"HumBERT, printed"
Sectors,0.790,0.768,0.780
Pillars 1D,0.705,0.686,0.700
Subpillars 1D,0.574,0.542,0.540
Pillars 2D,0.721,0.698,0.710
Subpillars 2D,0.527,0.502,0.510
Mean,0.663,0.639,0.648


In [10]:
results["gold"] = {
    "labels": sum(len(item.labels) for item in sample),
    "by_tier": {
        str(t): n
        for t, n in sorted(
            pd.Series([data.taxonomy.tiers[label] for item in sample for label in item.labels])
            .value_counts()
            .items()
        )
    },
}
results["rare_at_most"] = ex.rare_cutoff(data.train_counts)
version_1 = ex.load_humset(root)
used = {
    version: {label for item in d.train for label in item.labels}
    for version, d in (("2.0.0", data), ("1.0.0", version_1))
}
results["labels"] = {
    "in_use": len(used["2.0.0"]),
    "only_in_2.0.0": sorted(used["2.0.0"] - used["1.0.0"]),
    "only_in_1.0.0": sorted(used["1.0.0"] - used["2.0.0"]),
}
print("rare means at most", results["rare_at_most"], "training excerpts;", results["labels"])
# Version 1.0.0 keeps each source document's excerpts in one split; version 2.0.0 does not.
results["overlap"] = {
    "2.0.0": ex.split_overlap(data.train, data.test),
    "1.0.0": ex.split_overlap(version_1.train, version_1.test),
}
print("test excerpts sharing a source document, or their exact text, with training:", results["overlap"])
results["jobs_revision"] = gj.load_state(root)["revision"]
ex.save_results("15_humset_v2", results)
print("saved; LLM calls cost", results["llm_cost_usd"])

rare means at most 596 training excerpts; {'in_use': 77, 'only_in_2.0.0': ['Capacities & Response', 'Capacities & Response->People Reached/Response Gaps', 'Capacities & Response->Red Cross/Red Crescent', 'Casualties', 'Context', 'Covid-19', 'Covid-19->Prevention Campaign', 'Covid-19->Research And Outlook', 'Displacement', 'Humanitarian Access', 'Humanitarian Access->People Facing Humanitarian Access Constraints/Humanitarian Access Gaps', 'Humanitarian Conditions', 'Impact', 'Information And Communication', 'Priority Needs', 'Shock/Event', 'Shock/Event->Mitigating Factors'], 'only_in_1.0.0': ['Capacities & Response->Number Of People Reached/Response Gaps', 'Humanitarian Access->Number Of People Facing Humanitarian Access Constraints/Humanitarian Access Gaps']}
test excerpts sharing a source document, or their exact text, with training: {'2.0.0': {'same_document': 0.9693, 'same_text': 0.0228}, '1.0.0': {'same_document': 0.0, 'same_text': 0.0195}}
saved; LLM calls cost 8.34
